# Batch Normalization: Stabilizing Deep Network Training

## Overview
This notebook explores Batch Normalization (BatchNorm), a technique that revolutionized deep learning by making networks train faster, more stably, and with better performance.

## Learning Objectives
- Understand the problems that BatchNorm solves
- Learn the mathematical foundation of BatchNorm
- See how BatchNorm works in practice with CNNs
- Understand when and where to use BatchNorm
- Compare training with and without BatchNorm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

print("Batch Normalization: Making Deep Networks Train Better")
print("=" * 55)
print("Introduced by Ioffe & Szegedy (2015)")
print("One of the most important innovations in deep learning")

Batch Normalization: Making Deep Networks Train Better
Introduced by Ioffe & Szegedy (2015)
One of the most important innovations in deep learning


## Batch Normalization: Making Deep Networks Train Better
Introduced by [Ioffe & Szegedy (2015)](https://arxiv.org/pdf/1502.03167)
One of the most important innovations in deep learning

## The Problem:

Before BatchNorm, deep networks suffered from several training problems:

### 1. **📈 Numbers Get Too Big or Too Small (Unstable Gradients)**
- Too big numbers: The network "explodes" and gives wrong answers
- Too small numbers: The network can't learn anything
- Training becomes impossible: The network either crashes or learns nothing

### 2. **🎯 Very Sensitive to Starting Values (Initialization)**
- Wrong starting numbers: Training dies before it starts
- Had to guess perfect starting values: Very difficult and time-consuming
- Very slow learning: Had to use tiny learning rates to be safe

### 3. **🔄 Each Layer's Input Keeps Changing (Internal Covariate Shift)**
- Each layer must constantly adapt: No layer can master its job
- Very slow learning: Like trying to learn something while the rules keep changing
- Deeper networks fail: The more layers, the worse this problem gets

### 4. **😴 Neurons Stop Working (Saturated Activations)s**
- Stuck neurons: When output is always 1.0, neuron can't learn more
- Unresponsive neurons: When input is too small, neuron doesn't react
- Dead neurons: Half your network stops working


🔥 All Problems Combined = Training Disaster

## 🎯 The Core Problem
- The main issue: Deep networks are very sensitive to the scale and distribution of numbers flowing through them. Small changes get amplified, and the network becomes unstable.
- What researchers needed: A way to automatically keep numbers at the right scale throughout the entire network, no matter how deep.
- The solution: Batch Normalization - automatically standardize the numbers between each layer.

💡 Key Points:
- Numbers must stay in good range: Not too big, not too small
- Each layer needs consistent inputs: Same pattern every time
- Deep networks amplify problems: More layers = worse issues
- Manual tuning was impossible: Too many things to get exactly right

## The Solution: Batch Normalization

BatchNorm normalizes the inputs to each layer to have:
- **Mean = 0**
- **Standard deviation = 1**

But it's more sophisticated than simple normalization!

### The BatchNorm Formula:

For a mini-batch of inputs **x₁, x₂, ..., x_m**:

1. **Compute batch statistics:**
   ```
   μ = (1/m) ∑ xᵢ          # Batch mean
   σ² = (1/m) ∑ (xᵢ - μ)²   # Batch variance
   ```

2. **Normalize:**
   ```
   x̂ᵢ = (xᵢ - μ) / √(σ² + ε)
   ```
   (ε is a small constant for numerical stability)

3. **Scale and shift (learnable parameters):**
   ```
   yᵢ = γ * x̂ᵢ + β
   ```
   - **γ (gamma)**: learnable scale parameter
   - **β (beta)**: learnable shift parameter

### Why Scale and Shift?
- Pure normalization (mean=0, std=1) might be too restrictive
- The network can learn optimal mean and variance for each layer
- If γ=σ and β=μ, we recover the original distribution!

In [ ]:
# Implement BatchNorm from scratch to understand it
class BatchNorm1dFromScratch(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum

        # Learnable parameters
        self.gamma = nn.Parameter(torch.ones(num_features))   # Scale
        self.beta = nn.Parameter(torch.zeros(num_features))   # Shift

        # Running statistics (for inference)
        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))

    def forward(self, x):
        if self.training:
            # Training mode: use batch statistics
            batch_mean = x.mean(dim=0)
            batch_var = x.var(dim=0, unbiased=False)

            # Update running statistics
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * batch_mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * batch_var

            # Normalize using batch statistics
            x_normalized = (x - batch_mean) / torch.sqrt(batch_var + self.eps)
        else:
            # Inference mode: use running statistics
            x_normalized = (x - self.running_mean) / torch.sqrt(self.running_var + self.eps)

        # Scale and shift
        out = self.gamma * x_normalized + self.beta
        return out

# Test our implementation
torch.manual_seed(42)
batch_size, features = 32, 10
x = torch.randn(batch_size, features) * 3 + 5  # Mean≈5, Std≈3

bn = BatchNorm1dFromScratch(features)
bn.train()  # Training mode

print("Input statistics:")
print(f"  Mean: {x.mean(dim=0)[:3]} ...")
print(f"  Std:  {x.std(dim=0)[:3]} ...")

y = bn(x)
print("\nAfter BatchNorm:")
print(f"  Mean: {y.mean(dim=0)[:3]} ...")
print(f"  Std:  {y.std(dim=0)[:3]} ...")

print(f"\nLearnable parameters:")
print(f"  Gamma (scale): {bn.gamma[:3]} ...")
print(f"  Beta (shift):  {bn.beta[:3]} ...")

Input statistics:
  Mean: tensor([4.0267, 6.0704, 4.3642]) ...
  Std:  tensor([3.0043, 2.9457, 2.7858]) ...

After BatchNorm:
  Mean: tensor([ 6.7055e-08,  2.5146e-07, -3.7253e-08], grad_fn=<SliceBackward0>) ...
  Std:  tensor([1.0160, 1.0160, 1.0160], grad_fn=<SliceBackward0>) ...

Learnable parameters:
  Gamma (scale): tensor([1., 1., 1.], grad_fn=<SliceBackward0>) ...
  Beta (shift):  tensor([0., 0., 0.], grad_fn=<SliceBackward0>) ...


## BatchNorm in CNNs: BatchNorm2d

For convolutional layers, we use **BatchNorm2d**:

### Key Differences:
- **Input shape**: (Batch, Channels, Height, Width)
- **Normalization**: Applied per channel across batch and spatial dimensions
- **Parameters**: One γ and β per channel (not per pixel!)

### How it works:
1. For each channel, compute mean and variance across:
   - All samples in the batch
   - All spatial locations (H×W)
2. Normalize each pixel using channel statistics
3. Apply learnable scale (γ) and shift (β) per channel

### Typical CNN Block:
```
Conv2d → BatchNorm2d → ReLU → [Optional: Pooling]
```

In [ ]:
# Demonstrate BatchNorm2d with a CNN example
def demonstrate_batchnorm2d():
    # Create sample data: batch of RGB images
    batch_size, channels, height, width = 8, 3, 32, 32
    x = torch.randn(batch_size, channels, height, width)

    print(f"Input shape: {x.shape}")
    print(f"Input statistics per channel:")
    for c in range(channels):
        channel_data = x[:, c, :, :]
        print(f"  Channel {c}: mean={channel_data.mean():.3f}, std={channel_data.std():.3f}")

    # Apply BatchNorm2d
    bn2d = nn.BatchNorm2d(channels)
    y = bn2d(x)

    print(f"\nAfter BatchNorm2d:")
    for c in range(channels):
        channel_data = y[:, c, :, :]
        print(f"  Channel {c}: mean={channel_data.mean():.3f}, std={channel_data.std():.3f}")

    print(f"\nBatchNorm2d parameters:")
    print(f"  Gamma shape: {bn2d.weight.shape}  # One per channel")
    print(f"  Beta shape:  {bn2d.bias.shape}   # One per channel")
    print(f"  Running mean: {bn2d.running_mean.shape}")
    print(f"  Running var:  {bn2d.running_var.shape}")

demonstrate_batchnorm2d()

Input shape: torch.Size([8, 3, 32, 32])
Input statistics per channel:
  Channel 0: mean=0.010, std=1.002
  Channel 1: mean=-0.004, std=1.001
  Channel 2: mean=0.005, std=1.004

After BatchNorm2d:
  Channel 0: mean=-0.000, std=1.000
  Channel 1: mean=0.000, std=1.000
  Channel 2: mean=0.000, std=1.000

BatchNorm2d parameters:
  Gamma shape: torch.Size([3])  # One per channel
  Beta shape:  torch.Size([3])   # One per channel
  Running mean: torch.Size([3])
  Running var:  torch.Size([3])


## Training vs Inference Mode

BatchNorm behaves differently during training and inference:

### Training Mode:
- Computes mean and variance from current mini-batch
- Updates running statistics using momentum
- Adds noise (different statistics for each batch) → **regularization effect**

### Inference Mode:
- Uses running mean and variance (collected during training)
- Deterministic output (no batch dependency)
- Essential for single-sample inference

### Why Different Modes?
- During training: we have full batches to compute reliable statistics
- During inference: we might have only 1 sample (can't compute batch stats)
- Running statistics provide stable normalization for inference

In [ ]:
# Demonstrate training vs inference mode
def show_training_vs_inference():
    bn = nn.BatchNorm1d(5)
    x1 = torch.randn(10, 5) * 2 + 1  # Batch 1
    x2 = torch.randn(10, 5) * 3 - 2  # Batch 2 (different distribution)

    print("Training Mode (using batch statistics):")
    print("=" * 45)

    bn.train()  # Set to training mode

    y1 = bn(x1)
    print(f"Batch 1 input mean: {x1.mean(dim=0)[0]:.3f}")
    print(f"Batch 1 output mean: {y1.mean(dim=0)[0]:.3f}")

    y2 = bn(x2)
    print(f"\nBatch 2 input mean: {x2.mean(dim=0)[0]:.3f}")
    print(f"Batch 2 output mean: {y2.mean(dim=0)[0]:.3f}")

    print(f"\nRunning statistics after training:")
    print(f"Running mean: {bn.running_mean[0]:.3f}")
    print(f"Running var: {bn.running_var[0]:.3f}")

    print("\n" + "=" * 45)
    print("Inference Mode (using running statistics):")
    print("=" * 45)

    bn.eval()  # Set to inference mode

    # Single sample inference
    x_single = torch.randn(1, 5) * 10  # Very different distribution
    y_single = bn(x_single)

    print(f"Single sample input: {x_single[0, 0]:.3f}")
    print(f"Single sample output: {y_single[0, 0]:.3f}")
    print("Note: Uses running statistics, not single-sample stats")

show_training_vs_inference()

Training Mode (using batch statistics):
Batch 1 input mean: 1.658
Batch 1 output mean: 0.000

Batch 2 input mean: -2.503
Batch 2 output mean: -0.000

Running statistics after training:
Running mean: -0.101
Running var: 2.667

Inference Mode (using running statistics):
Single sample input: -0.090
Single sample output: 0.007
Note: Uses running statistics, not single-sample stats


## CNN with BatchNorm: Complete Example

Let's build a CNN with and without BatchNorm to see the difference:

In [ ]:
# CNN without BatchNorm
class CNNWithoutBN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(128 * 4 * 4, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 32x32 → 16x16
        x = self.pool(F.relu(self.conv2(x)))  # 16x16 → 8x8
        x = self.pool(F.relu(self.conv3(x)))  # 8x8 → 4x4
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# CNN with BatchNorm
class CNNWithBN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(128 * 4 * 4, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # Conv → BN → ReLU → Pool
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Compare the models
model_without_bn = CNNWithoutBN()
model_with_bn = CNNWithBN()

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model Comparison:")
print(f"Without BatchNorm: {count_parameters(model_without_bn):,} parameters")
print(f"With BatchNorm:    {count_parameters(model_with_bn):,} parameters")
print(f"\nBatchNorm adds: {count_parameters(model_with_bn) - count_parameters(model_without_bn)} parameters")
print("(2 parameters per channel: γ and β)")

# Test forward pass
x = torch.randn(4, 3, 32, 32)  # Batch of 4 RGB images
print(f"\nInput shape: {x.shape}")
print(f"Output without BN: {model_without_bn(x).shape}")
print(f"Output with BN:    {model_with_bn(x).shape}")

Model Comparison:
Without BatchNorm: 113,738 parameters
With BatchNorm:    114,186 parameters

BatchNorm adds: 448 parameters
(2 parameters per channel: γ and β)

Input shape: torch.Size([4, 3, 32, 32])
Output without BN: torch.Size([4, 10])
Output with BN:    torch.Size([4, 10])


## Benefits of Batch Normalization

### 1. **Faster Training**
- Can use higher learning rates (10-100x higher!)
- Converges in fewer epochs
- Less sensitive to learning rate choice

### 2. **Better Gradient Flow**
- Prevents vanishing/exploding gradients
- More stable training dynamics
- Enables deeper networks

### 3. **Regularization Effect**
- Noise from batch statistics acts as regularization
- Can reduce or eliminate dropout
- Better generalization

### 4. **Less Sensitive to Initialization**
- Network is more robust to weight initialization
- Can start training with poor initial weights
- More forgiving hyperparameter choices

### 5. **Improved Performance**
- Often achieves better final accuracy
- More consistent results across runs
- Essential for state-of-the-art models

## When and Where to Use BatchNorm

### ✅ **When to Use:**
- Almost all modern CNN architectures
- Deep networks (>3-4 layers)
- When training stability is important
- When you want faster training

### ❌ **When NOT to Use:**
- Very small batch sizes (batch_size < 4)
- Online learning (batch_size = 1)
- Some specific architectures (e.g., some GANs)
- When batch statistics don't make sense

### 📍 **Where to Place:**
- **Typical**: Conv2d → BatchNorm2d → ReLU
- **Alternative**: Conv2d → ReLU → BatchNorm2d
- **After** linear/conv layers, **before** activation functions
- Don't use before final output layer

### 🔧 **Best Practices:**
- Use momentum=0.1 (default is usually good)
- Always call model.eval() during inference
- Can often reduce/remove dropout when using BatchNorm
- Works well with most optimizers (Adam, SGD, etc.)

## Summary

Batch Normalization is one of the most important innovations in deep learning:

### **Key Takeaways:**
1. **Problem**: Deep networks suffer from internal covariate shift and training instability
2. **Solution**: Normalize layer inputs to have mean=0, std=1, then learn optimal scale/shift
3. **Formula**: y = γ × (x - μ)/σ + β
4. **Benefits**: Faster training, higher learning rates, better stability, regularization
5. **Usage**: Place after Conv/Linear layers, before activation functions

### **Modern CNN Pattern:**
```
Conv2d → BatchNorm2d → ReLU → [Optional: Pooling]
```

### **Training vs Inference:**
- **Training**: Use batch statistics + update running stats
- **Inference**: Use running statistics for consistency

BatchNorm has become so essential that it's included in virtually every modern CNN architecture (ResNet, DenseNet, EfficientNet, etc.). It's a perfect example of how a simple idea can have profound impact on the field!

---